# Plant Disease Detection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Rescaling

In [ ]:
# Define standard image size and batch size
img_height = 128
img_width = 128
batch_size = 32
data_dir = "plant_dataset" # Make sure your extracted dataset folder is named this

print("Loading Training Data:")
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

print("\nLoading Validation Data:")
val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

# Extract the class names (the names of your subfolders)
class_names = train_ds.class_names
print(f"\nClasses found: {class_names}")
num_classes = len(class_names)

In [ ]:
model = Sequential()

# 1. Normalization Layer
model.add(Rescaling(1./255, input_shape=(img_height, img_width, 3)))

# 2. First Convolutional Block
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# 3. Second Convolutional Block
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# 4. Third Convolutional Block
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# 5. Flatten and Dense Layers
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=10
)

In [ ]:
# Grab one batch of images and labels from the validation set
for images, labels in val_ds.take(1):
    test_image = images[0]
    actual_label = labels[0]
    
    # Model expects a batch, so we expand dimensions
    img_array = tf.expand_dims(test_image, 0) 
    
    # Make prediction
    predictions = model.predict(img_array)
    predicted_class = np.argmax(predictions[0])
    
    # Display the image
    plt.imshow(test_image.numpy().astype("uint8"))
    plt.title(f"Actual: {class_names[actual_label]}\nPredicted: {class_names[predicted_class]}")
    plt.axis("off")
    plt.show()